# Adaptive Basket Recommender: Run and Test in Google Colab

This notebook checks that the adaptive system works and lets you try it. Run the cells **in order, top to bottom**. No GPU is needed.

| Step | What it does |
|---|---|
| 1 | Loads the project (from GitHub, or from the ZIP file you upload) |
| 2 | Installs the libraries it needs |
| 3 | Runs the 30 automated tests |
| 4 | Runs the simulation and draws the learning curve |
| 5 | Replays a scripted session that shows the system adapting |
| 6 | Point-and-click interface: answer Yes or No and watch the suggestions change |
| 7 | Lets you chat with the assistant yourself |

## 1. Load the project

**Option A, from GitHub:** paste the repository URL between the quotes (the repository must be public).

**Option B, from the ZIP:** leave `REPO_URL` empty. When you run the cell, a **Choose Files** button appears; select `adaptive-basket-recommender.zip`.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/Royphilips/adaptive-basket-recommender.git"

import os, subprocess, sys, zipfile, glob

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE = "/content" if IN_COLAB else os.path.abspath(".")
PROJECT_DIR = os.path.join(BASE, "adaptive-basket-recommender")

if os.path.exists(os.path.join(PROJECT_DIR, "adaptive")):
    print("Project already loaded.")
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT_DIR], check=True)
    print("Cloned", REPO_URL)
elif IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # pick adaptive-basket-recommender.zip
    zip_name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(BASE)
    if not os.path.exists(os.path.join(PROJECT_DIR, "adaptive")):
        # Accept a ZIP whose top folder has a different name, or files nested one level down
        found = [f for f in glob.glob(os.path.join(BASE, "**", "adaptive", "engine.py"), recursive=True)
                 if "__MACOSX" not in f]
        if not found:
            raise FileNotFoundError("Couldn't find the project inside the ZIP.")
        PROJECT_DIR = os.path.dirname(os.path.dirname(found[0]))
    print("Unzipped", zip_name)
else:
    raise FileNotFoundError("Set REPO_URL or put the adaptive-basket-recommender folder next to this notebook.")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("Working in", os.getcwd())
print(sorted(os.listdir(".")))

## 2. Install the libraries

Colab already has pandas, pytest, matplotlib and ipywidgets. This makes sure mlxtend is recent enough for the starting system's tests.

In [ ]:
%pip install -q "mlxtend>=0.23.2" tabulate
import mlxtend, pandas
print("Python", sys.version.split()[0], "| mlxtend", mlxtend.__version__, "| pandas", pandas.__version__)

## 3. Run the automated tests

You should see **30 passed**: 13 tests from the starting system and 17 for the adaptive layer (cautious learning, hiding and un-hiding items, fading evidence, undo, reset, saved profiles, and that switching adaptation off gives exactly the starting system's suggestions).

In [ ]:
!python -m pytest -q -p no:cacheprovider

## 4. Run the simulation

300 simulated shoppers with hidden likes and dislikes use the starting system and the adaptive system. Expected result for trips 7–12: about **1.90** suggestions accepted per trip with the starting system and **2.25** with the adaptive system. The shoppers are generated from fixed seeds, so your numbers should match.

In [ ]:
!python simulate.py | tail -4

In [ ]:
from IPython.display import Image, display
display(Image("results/learning_curve.png", width=820))

## 5. Watch it adapt (scripted session)

A new shopper makes four taco-night trips and keeps turning down avocado and other produce. Watch for the lines that start with **Note:**, which is the system telling the shopper that it has changed its behavior, and for the suggestion list changing between trips.

In [ ]:
import tempfile
from pathlib import Path
from adaptive import UserProfile
from adaptive_cli import AdaptiveAssistant, greeting
from recommender import BasketRecommender, load_catalog, load_transactions

model = BasketRecommender().fit(load_transactions(), load_catalog())
demo_folder = Path(tempfile.mkdtemp())            # a throwaway profile folder for this demo
bot = AdaptiveAssistant(model, UserProfile(name="demo"), demo_folder)
print(greeting(bot), "\n")

script = ["I have tortillas and ground beef", "no avocado", "yes salsa", "checkout",
          "add tortillas, ground beef", "no avocado", "no lettuce", "yes salsa", "checkout",
          "add tortillas, ground beef", "no avocado", "checkout",
          "add tortillas, ground beef", "why salsa", "why cheddar cheese", "profile",
          "adapt off", "adapt on", "allow avocado"]
for message in script:
    print(">", message)
    print(bot.handle(message), "\n")

## 6. Point-and-click interface

Pick items for your basket, then answer the suggestions with **Yes** or **No**. Press **Checkout** to finish a trip; after two or more trips the system starts offering your usuals. The panel at the bottom always shows what the system has learned about you, and **Forget everything** wipes it.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

gui_folder = Path(tempfile.mkdtemp())
gui = AdaptiveAssistant(model, UserProfile(name="colab"), gui_folder)

picker = widgets.Dropdown(options=model.items, description="Item")
add_btn = widgets.Button(description="Add to basket", button_style="primary")
checkout_btn = widgets.Button(description="Checkout")
undo_btn = widgets.Button(description="Undo")
reset_btn = widgets.Button(description="Forget everything", button_style="warning")
adapt_box = widgets.Checkbox(value=True, description="Adaptation on")
basket_label = widgets.HTML()
notes = widgets.HTML()
rows = widgets.VBox()
learned = widgets.HTML()

def run(command):
    reply = gui.handle(command)
    msgs = [line for line in reply.splitlines() if line.startswith(("Note:", "Trip ", "Undone", "I've forgotten"))]
    notes.value = "".join(f"<div style='margin:4px 0'><b>{m}</b></div>" for m in msgs)
    refresh()

def refresh():
    gui.suggestions()                               # updates the list on screen
    basket_label.value = "<b>Basket:</b> " + (", ".join(sorted(gui.basket)) or "(empty)")
    details = gui.profile.wants_details()
    new_rows = []
    for rec in gui.last_shown:
        yes = widgets.Button(description="Yes", button_style="success", layout=widgets.Layout(width="60px"))
        no = widgets.Button(description="No", layout=widgets.Layout(width="60px"))
        yes.on_click(lambda _, item=rec.item: run(f"yes {item.lower()}"))
        no.on_click(lambda _, item=rec.item: run(f"no {item.lower()}"))
        new_rows.append(widgets.HBox([yes, no, widgets.Label(rec.explain(details))]))
    rows.children = new_rows
    learned.value = "<br>".join(gui.profile.summary())

add_btn.on_click(lambda _: run(f"add {picker.value.lower()}"))
checkout_btn.on_click(lambda _: run("checkout"))
undo_btn.on_click(lambda _: run("undo"))
reset_btn.on_click(lambda _: run("reset profile"))
adapt_box.observe(lambda change: run("adapt on" if change["new"] else "adapt off"), names="value")

refresh()
display(widgets.VBox([
    widgets.HBox([picker, add_btn, checkout_btn, undo_btn]),
    basket_label, notes,
    widgets.HTML("<b>Suggestions</b>"), rows,
    widgets.HTML("<hr><b>What I've learned about you</b>"), learned,
    widgets.HBox([adapt_box, reset_btn]),
]))

## 7. Chat with the assistant yourself

Run the cell and type in the box that appears. Try *I have pasta and parmesan*, then answer with `yes 1` or `no 2`, and `checkout` when you finish a trip. Other commands: `profile`, `why salsa`, `undo`, `adapt off`, `details on`, `reset profile`, `help`.

Type **quit** to finish. The cell keeps running until you do.

In [ ]:
chat = AdaptiveAssistant(model, UserProfile.load("colab-chat", Path(PROJECT_DIR) / "profiles"), Path(PROJECT_DIR) / "profiles")
if IN_COLAB:
    print(greeting(chat))
    while True:
        message = input("> ")
        if message.strip().lower() in {"quit", "exit", "bye"}:
            print("Goodbye! Your profile is saved.")
            break
        print(chat.handle(message))
else:
    print("Interactive chat runs in Colab; outside it, see the scripted session in step 5.")

## Done

If the tests showed **30 passed**, the simulation reported about **1.90 vs 2.25**, and the suggestions in steps 5 and 6 changed in response to the answers, the adaptive system is working.